# Max-`dlambda` Ablation

Sweeps Newton load-step size with optional reaction-force loss and optional Hessian regularization.

In [ ]:
import json
import os

from properties import SlinkyN3Properties
from run_max_dlambda_ablation_minimal import (
    MaxDlambdaAblationConfig,
    run_max_dlambda_ablation,
    summarize_best_stable_step,
    subset_energy_only,
    subset_main_paper_candidates,
    subset_all,
)


In [ ]:
# Dataset / physical setup
properties = SlinkyN3Properties(mass=0.3)
train_file = "../simulation_data_2D/3_noded/n3_slinky_sim_train_dataset.npz"
valid_file = "../simulation_data_2D/3_noded/n3_slinky_sim_test_dataset.npz"

# selected_architectures = subset_energy_only()
# selected_architectures = subset_main_paper_candidates()
selected_architectures = subset_all()

# Optional force loss. Leave strength at 0.0 for displacement-only training.
force_loss_strength = 1.0
force_key = None
force_components = (0,)
force_sign = 1.0
return_loss_components = force_loss_strength != 0.0

# Optional Hessian regularizer.
hessian_reg_strength = 0.0
hessian_reg_probes = 1
hessian_reg_seed = 0


In [ ]:
cfg = MaxDlambdaAblationConfig(
    output_dir="max_dlambda_ablation_outputs_n3_slinky_simdata_force_optional",
    n_epochs=300,
    lr=1e-2,
    seed_list=(42,),
    hidden=(10,),
    input_mode="invariant",
    activation="tanh",
    corr_factor=0.01,
    only_stretching_NN=False,
    zero_reference=True,
    max_dlambda_values=(1e-3, 5e-3, 1e-2, 5e-2, 1e-1),
    valid_every=1,
    iters=20,
    ls_steps=10,
    abs_tol=1e-4,
    rel_tol=1e-4,
    train_fail_on_nonconvergence=True,
    prediction_fail_on_nonconvergence=False,
    hessian_reg_strength=hessian_reg_strength,
    hessian_reg_probes=hessian_reg_probes,
    hessian_reg_seed=hessian_reg_seed,
    force_key=force_key,
    force_loss_strength=force_loss_strength,
    force_components=force_components,
    force_sign=force_sign,
    return_loss_components=return_loss_components,
    save_npz=True,
    save_model=False,
    save_summary_json=True,
    strict_finite_check=True,
    stop_after_first_failure=True,
    verbose=True,
)


In [ ]:
results = run_max_dlambda_ablation(
    properties=properties,
    train_file=train_file,
    valid_file=valid_file,
    cfg=cfg,
    selected_architectures=selected_architectures,
)

stable_summary = summarize_best_stable_step(results)
stable_summary


In [ ]:
summary_path = os.path.join(cfg.output_dir, "best_stable_step_summary.json")
with open(summary_path, "w") as f:
    json.dump(stable_summary, f, indent=2)
summary_path
